In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

reponse = llm.invoke("Dis bonjour en une phrase.")

print("RÉPONSE :", reponse.content)
print("RAISONNEMENT :", reponse.additional_kwargs.get("reasoning_content", "(aucun)")[:200])

RÉPONSE : Bonjour !
RAISONNEMENT : Okay, the user asked "Dis bonjour en une phrase." which translates to "Say hello in one sentence." in French.

First, I need to respond with a greeting in French. The phrase "bonjour" is the French wo


In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte."
)

In [5]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Je m'appelle Houssam"}]
})
print(response["messages"][-1].content)

Bonjour Houssam ! 😊


In [6]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "Quel est mon prénom ?"}]
})
print(response["messages"][-1].content)

Je ne sais pas votre prénom.


In [7]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français, de façon courte.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "1"}}

In [8]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Je m'appelle Houssam"}]},
    config=config
)
print(res["messages"][-1].content)

Bonjour Houssam ! 😊


In [9]:
res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config
)
print(res["messages"][-1].content)

Houssam 😊


In [10]:
config2 = {"configurable": {"thread_id": "2"}}

res = agent_memoire.invoke(
    {"messages": [{"role": "user", "content": "Quel est mon prénom ?"}]},
    config=config2
)
print(res["messages"][-1].content)

Je ne sais pas votre prénom.


## Étape 2 : Tools personnalisés

In [12]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Donne la météo actuelle d'une ville."""
    print(f"[TOOL] get_weather appelé pour {location}")
    return f"Météo à {location} : ensoleillé, 32°C"

@tool
def get_employee_info(name: str) -> dict:
    """Donne le salaire et le poste d'un employé à partir de son prénom."""
    print(f"[TOOL] get_employee_info appelé pour {name}")
    return {"name": name, "salary": 12000, "job": "Développeur"}

@tool
def multiply(a: float, b: float) -> float:
    """Multiplie deux nombres."""
    print(f"[TOOL] multiply appelé avec a={a}, b={b}")
    return a * b

tools = [get_weather, get_employee_info, multiply]

In [13]:
print(get_weather.invoke({"location": "Marrakech"}))
print(multiply.invoke({"a": 6, "b": 7}))
print(get_weather.name, "-", get_weather.description)

[TOOL] get_weather appelé pour Marrakech
Météo à Marrakech : ensoleillé, 32°C
[TOOL] multiply appelé avec a=6.0, b=7.0
42.0
get_weather - Donne la météo actuelle d'une ville.


In [14]:
from langgraph.checkpoint.memory import InMemorySaver

agent_tools = create_agent(
    model=llm,
    tools=tools,
    system_prompt="Tu es un assistant utile. Utilise les outils quand c'est nécessaire. Réponds en français.",
    checkpointer=InMemorySaver()
)

config_tools = {"configurable": {"thread_id": "tools-1"}}

In [15]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quelle est la météo à Casablanca ?"}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_weather appelé pour Casablanca
La météo à Casablanca est actuellement ensoleillée à 32°C.


In [16]:
res = agent_tools.invoke(
    {"messages": [{"role": "user", "content": "Quel est le salaire de Mohamed ? Multiplie-le par 3."}]},
    config=config_tools
)
print(res["messages"][-1].content)

[TOOL] get_employee_info appelé pour Mohamed
[TOOL] multiply appelé avec a=12000.0, b=3.0
Le salaire de Mohamed est de 12 000. Multiplié par 3, cela donne 36 000.
